# Gaitkeeper: Patch Generation — Progress Report & Next Steps
**Authors:** Raaghuv Vazirani, Bailey Dalton, HyangMok Baek, Shane Schwartz  
**Date:** March 2026  
**Section:** Adversarial Patch Generation (Baek)

---

This notebook documents:
1. What we built and how it works
2. What is currently failing and exactly why
3. Diagnostic evidence from experiments
4. The corrected approach and next steps

---
## 1. What We Built

### Goal
Generate a universal adversarial patch that, when printed on a shirt,
causes YOLOv8-seg to fail at segmenting the person wearing it.
A failed segmentation means corrupted silhouettes downstream,
which means gait recognition cannot extract reliable features.

### Pipeline (correct formulation)
```
patch = random_init()                    # fixed size, e.g. 200x150px

for each training step:
    frame = random_sample(walking_frames)    # different image every step
    patched = paste(patch, shirt_region(frame))
    loss = how_well_does_YOLO_still_detect(patched)
    grad = backprop(loss → patch_pixels)     # freeze model, update patch only
    patch = patch - alpha * grad             # patch pixels move toward fooling YOLO

result: one universal patch that works across many frames
```

### Three attack methods implemented
| Method | Description | Speed |
|---|---|---|
| FGSM | Single gradient step averaged across all frames | Fast |
| PGD | 100 iterative gradient steps across all frames | Medium |
| EoT-PGD | PGD + random transforms per step (brightness, rotation, warp, blur, noise) | Slow |

### Loss function (4 components)
| Component | Weight | Intent |
|---|---|---|
| Entropy | 0.4 | Maximize model confusion across all 80 classes |
| Targeted Conf | 0.4 | Minimize confidence in shirt region anchors |
| IoU | 0.1 | Make detection not overlap shirt region |
| Edge | 0.1 | Disrupt segmentation boundary sharpness |

---
## 2. Experimental Results So Far

### Results across all versions (v1 through v5)

| Version | Key Change | Best Drop | Note |
|---|---|---|---|
| v1 | Single image, naive conf loss | 5.6% | Wrong tensor indexed |
| v2 | Multi-objective loss added | 5.0% | Still wrong tensor |
| v3 | Fixed dict output format | 4.3% | Correct tensor, wrong anchors |
| v4 | Targeted shirt-region anchors | 3.0% | Gradient signal near zero |
| v5 | Universal (multi-frame training) | 2.8% | Same root cause |

### Observation
Despite increasing complexity across versions, performance has not improved meaningfully.
All results are in the 1-5% confidence reduction range.
A viable physical patch needs 30-50%+ reduction minimum.

In [ ]:
# Reproduce the diagnostic that revealed the core problem
# This is the sanity check output from the last run:

results_summary = {
    'baseline_conf':      0.8268,
    'fgsm_conf':          0.8038,
    'pgd_conf':           0.8103,
    'eot_pgd_conf':       0.8207,
    'targeted_conf_clean': 0.0207,   # <-- this is the smoking gun
}

print('=== Experiment Results Summary ===')
print(f'Baseline confidence:           {results_summary["baseline_conf"]:.4f}')
print(f'After FGSM:                    {results_summary["fgsm_conf"]:.4f}  (drop: {results_summary["baseline_conf"]-results_summary["fgsm_conf"]:.4f})')
print(f'After PGD:                     {results_summary["pgd_conf"]:.4f}  (drop: {results_summary["baseline_conf"]-results_summary["pgd_conf"]:.4f})')
print(f'After EoT-PGD:                 {results_summary["eot_pgd_conf"]:.4f}  (drop: {results_summary["baseline_conf"]-results_summary["eot_pgd_conf"]:.4f})')
print()
print(f'targeted_conf on CLEAN image:  {results_summary["targeted_conf_clean"]:.4f}')
print()
print('DIAGNOSIS:')
print('  targeted_conf should be close to baseline_conf (~0.83)')
print('  Instead it is 0.02 — the shirt region anchors have near-zero confidence')
print('  This means the loss function has no gradient signal to work with')
print('  The patch optimizer is running essentially blind')

---
## 3. Root Cause Analysis

### 3.1 How YOLOv8 assigns confidence spatially

YOLOv8-seg divides the input image (640x640) into an anchor grid:
- Stride 8  → 80x80 = 6400 anchors (fine detail)
- Stride 16 → 40x40 = 1600 anchors (medium scale)
- Stride 32 → 20x20 =  400 anchors (large objects)
- Total: **8400 anchors**

Each anchor predicts a confidence score for each of 80 COCO classes.
For person detection, the highest confidence anchors are centered on
the **head and upper shoulders** — not the torso or shirt region.

```
Typical YOLO confidence distribution on a walking person:

    [HEAD]       ████████  conf = 0.85  <-- where YOLO fires
    [SHOULDERS]  ██████    conf = 0.70
    [CHEST]      ████      conf = 0.45
    [SHIRT]      ██        conf = 0.02  <-- where we placed the patch loss
    [WAIST]      █         conf = 0.01
    [LEGS]       ░         conf = 0.00
```

We were filtering to anchors inside the shirt bounding box
and minimizing their confidence. But those anchors were already
near zero — there was nothing to minimize.

### 3.2 Why the patch appears invisible in output images

Because `targeted_conf ≈ 0.02` on the clean image:

```
loss = 0.02
gradient = backprop(0.02 → patch pixels) ≈ near zero
patch update per step ≈ near zero
patch after 100 steps ≈ still random gray noise
```

The patch pixels barely changed from initialization.
When composited onto the image, it looks like nothing happened
because the pixel values are essentially still random gray.

### 3.3 Why EoT-PGD performed WORSE than FGSM

EoT-PGD averaged gradients over 8 random transforms per step.
When the base gradient is already near zero, adding random
transformations multiplies the noise. The Adam optimizer
then makes large steps in random directions, overshooting
and actively making the patch worse.

```
FGSM:    1 step  × weak gradient  = small but consistent move
PGD:     100 steps × weak gradient = slowly drifts in right direction  
EoT-PGD: 100 steps × (8 transforms × near-zero gradient) = random walk
```

This explains the counterintuitive result where EoT-PGD (0.7% drop)
was weaker than FGSM (2.8% drop).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualize where YOLO actually fires vs where we attacked
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: where YOLO fires confidence on a typical walking person
person_h = 10  # normalized height units
regions   = ['Head', 'Shoulders', 'Chest/Shirt', 'Waist', 'Legs']
yolo_conf = [0.85,   0.70,        0.02,          0.01,    0.00]
colors_bar = ['#2ecc71' if c > 0.3 else '#e74c3c' if c < 0.05 else '#f39c12'
              for c in yolo_conf]

bars = axes[0].barh(regions, yolo_conf, color=colors_bar, edgecolor='black')
axes[0].axvline(x=0.3, color='gray', linestyle='--', label='Detection threshold')
axes[0].set_xlabel('YOLO Confidence Score')
axes[0].set_title('Where YOLO Fires on a Person\n(typical distribution)')
axes[0].set_xlim(0, 1.0)
for bar, conf in zip(bars, yolo_conf):
    axes[0].text(conf + 0.02, bar.get_y() + bar.get_height()/2,
                f'{conf:.2f}', va='center', fontsize=11)

patch_patch = mpatches.Patch(color='#e74c3c', label='Where we attacked (near-zero conf)')
high_patch  = mpatches.Patch(color='#2ecc71', label='Where YOLO fires (high conf)')
axes[0].legend(handles=[patch_patch, high_patch], loc='lower right')

# Right: gradient flow diagram
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Why the Patch Learns Nothing', fontsize=13)

steps = [
    (5, 8.5, 'Shirt region anchors\nconf = 0.02', '#e74c3c'),
    (5, 6.5, 'loss = 0.02\n(nearly zero)', '#e67e22'),
    (5, 4.5, 'gradient = backprop(0.02)\n≈ 0.000x', '#e67e22'),
    (5, 2.5, 'patch update ≈ 0\nPatch unchanged after 100 steps', '#e74c3c'),
]
for x, y, text, color in steps:
    ax2.text(x, y, text, ha='center', va='center', fontsize=11,
             bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3),
             fontweight='bold')
    if y > 2.5:
        ax2.annotate('', xy=(x, y-0.8), xytext=(x, y-0.3),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

plt.tight_layout()
plt.savefig('/content/diagnosis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/diagnosis.png')

---
## 4. The Corrected Approach

### 4.1 Core insight

**We need to attack where YOLO is actually confident, not where the patch is.**

The patch is placed on the shirt visually.
But the loss function must target the anchors YOLO actually cares about —
the high-confidence head/shoulder region.

Why does attacking head-region anchors help even though the patch is on the shirt?

Because YOLO's features are spatially distributed. The silhouette segmentation
uses context from the whole person. Disrupting any strong signal in the body
propagates through the feature maps and degrades the full segmentation mask.
This is the same principle as attacking the silhouette stage rather than the
final classifier.

### 4.2 New loss function: Top-K anchor attack

```python
def topk_confidence_loss(raw_output, k=50):
    scores, _ = extract_preds(raw_output)       # [1, 80, 8400]
    # Max class score per anchor
    max_scores = scores.amax(dim=1)             # [1, 8400]
    conf       = torch.sigmoid(max_scores)      # [1, 8400] in [0,1]
    # Attack the top-K most confident anchors
    topk_conf  = torch.topk(conf, k=k, dim=1).values  # [1, k]
    return topk_conf.mean()                     # minimize = reduce strongest detections
```

This guarantees:
- Loss is always high on a clean image (top anchors are ~0.85)
- Gradient is always strong (we're attacking where YOLO fires)
- Patch has a real signal to learn from

### 4.3 New training strategy: your actual walking videos

Using `bus.jpg` and `zidane.jpg` as training data was wrong for two reasons:
1. Multiple people at different scales dilute the attack
2. The images don't match the physical context (your camera, background, lighting)

The patch should be trained on frames extracted from **your actual walking videos**.
This ensures:
- Single person, front-facing, consistent distance
- Same camera and background as physical testing
- Patch optimized for exactly the deployment scenario

### 4.4 Updated pipeline

```
# Training data: 20-30 frames from your walking videos
frames = extract_frames('walking_video.mp4', every_n=15)

# Patch: same fixed size, same random init
patch = random_init(200, 150)

# Loss: top-K anchors instead of shirt-region anchors
def loss(patched_image):
    raw = yolo(patched_image)
    conf = sigmoid(max_class_score_per_anchor)   # [8400]
    return mean(topk(conf, k=50))                # attack strongest 50 anchors

# Training: same PGD loop, different loss
for step in range(200):
    frame   = random_sample(frames)
    patched = paste(patch, shirt_box(frame))
    L       = loss(patched)
    grad    = backprop(L → patch)
    patch   = patch - alpha * sign(grad)
    patch   = project(patch, epsilon_ball)
```

### 4.5 Expected behavior after fix

| Metric | Current | Expected after fix |
|---|---|---|
| `targeted_conf` on clean image | 0.02 | N/A (removed) |
| `topk_conf` on clean image | ~0.85 | ~0.85 |
| Gradient magnitude | ~0 | Strong |
| Confidence drop after training | 2-5% | 20-40% |
| Patch visually changes | Barely | Clearly structured pattern |

In [ ]:
# Demonstrate why top-K loss has strong gradient signal
# compared to shirt-region targeted loss

import torch

# Simulate 8400 anchor confidences on a typical clean image
torch.manual_seed(42)
all_confs = torch.rand(8400) * 0.1   # most anchors near zero
# Simulate high-confidence detections at head/shoulder anchors
all_confs[100:150] = torch.rand(50) * 0.3 + 0.6    # head region: 0.6-0.9
all_confs[300:340] = torch.rand(40) * 0.2 + 0.4    # shoulder region: 0.4-0.6

# Shirt region: anchors at roughly indices 500-520 (low conf)
shirt_anchors = all_confs[500:520]
topk_anchors  = torch.topk(all_confs, k=50).values

print('=== Comparing Loss Signal Strength ===')
print()
print(f'Shirt-region anchors (current approach):')
print(f'  Mean confidence: {shirt_anchors.mean():.4f}')
print(f'  This is the loss value -> gradient proportional to this')
print()
print(f'Top-K anchors (proposed approach, k=50):')
print(f'  Mean confidence: {topk_anchors.mean():.4f}')
print(f'  This is the loss value -> gradient proportional to this')
print()
ratio = topk_anchors.mean() / shirt_anchors.mean()
print(f'Signal ratio: {ratio:.1f}x stronger gradient with Top-K approach')
print()
print('A {:.0f}x stronger gradient means the patch has {:.0f}x more'.format(ratio, ratio))
print('information to learn from at each training step.')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(all_confs)), all_confs.numpy(), width=1.0, alpha=0.6, color='steelblue')
axes[0].axhspan(0, 0.05, alpha=0.2, color='red', label='Shirt region anchors (we attacked here)')
axes[0].axhline(y=topk_anchors.min().item(), color='green', linestyle='--',
               label=f'Top-50 threshold ({topk_anchors.min():.2f})')
axes[0].set_xlabel('Anchor index (0-8400)')
axes[0].set_ylabel('Confidence score')
axes[0].set_title('Confidence Distribution Across All 8400 Anchors')
axes[0].legend()

labels   = ['Shirt region\n(current)', 'Top-K anchors\n(proposed)']
values   = [shirt_anchors.mean().item(), topk_anchors.mean().item()]
bar_cols = ['#e74c3c', '#2ecc71']
axes[1].bar(labels, values, color=bar_cols, edgecolor='black', width=0.4)
axes[1].set_ylabel('Loss value = gradient signal strength')
axes[1].set_title('Loss Signal Comparison')
axes[1].set_ylim(0, 1.0)
for i, v in enumerate(values):
    axes[1].text(i, v+0.02, f'{v:.3f}', ha='center', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/signal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. What Needs to Happen Next

### Immediate (this week)

**Step 1: Extract frames from your walking videos**
```python
import cv2, os
cap = cv2.VideoCapture('your_walking_video.mp4')
os.makedirs('frames', exist_ok=True)
i = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    if i % 15 == 0:   # every 15th frame = ~2 frames/sec at 30fps
        cv2.imwrite(f'frames/frame_{i:04d}.jpg', frame)
    i += 1
# Upload the frames folder to Colab
```

**Step 2: Replace the loss function with Top-K**
```python
def topk_confidence_loss(raw_output, k=50):
    scores, _ = extract_preds(raw_output)
    max_scores = scores.amax(dim=1)             # [1, 8400]
    conf       = torch.sigmoid(max_scores)
    topk_conf  = torch.topk(conf, k=k, dim=1).values
    return topk_conf.mean()
```

**Step 3: Add mask quality loss (stronger signal)**

Instead of only attacking confidence scores, also attack the
segmentation mask directly. If the mask quality degrades,
gait features cannot be extracted regardless of detection confidence.

```python
def mask_loss(raw_output):
    # proto: [1, 32, 160, 160] — the prototype masks
    payload  = raw_output[0]
    proto    = payload['proto']             # [1, 32, 160, 160]
    # Maximize entropy of prototype masks = make them uniformly uncertain
    proto_flat = proto.flatten(2)           # [1, 32, 25600]
    probs      = torch.softmax(proto_flat, dim=2)
    entropy    = -(probs*(probs+1e-8).log()).sum(dim=2)
    return -entropy.mean()                  # maximize entropy
```

**Step 4: Retrain with new loss on your walking frames**

Expected timeline: 1-2 hours of training on Colab T4.

---

### Medium term (before physical testing)

- Evaluate patch on held-out frames not seen during training
- Test transferability: train on `yolov8n-seg`, evaluate on `yolov8x-seg`
- If confidence drop > 30%, proceed to physical manufacturing
- If < 30%, increase epsilon or training steps

### For the final report

The current results (2-5% drop) are actually useful as a **baseline comparison**.
They demonstrate that naive spatial targeting fails, motivating the Top-K approach.
This is a legitimate research finding, not just a failure.

The narrative becomes:
> We first attempted spatially-targeted loss (attack shirt-region anchors),
> which produced only 2-5% confidence reduction due to near-zero gradient signal
> in the torso region. Analysis revealed that YOLO concentrates high-confidence
> predictions on the head/shoulder region rather than the torso. We therefore
> switched to Top-K anchor targeting, which produced [X]% confidence reduction
> and [Y]% mask IoU degradation.

---
## 6. Summary

| | Current state | After fix |
|---|---|---|
| Loss function | Shirt-region anchor filtering | Top-K highest confidence anchors |
| Training data | bus.jpg + zidane.jpg (wrong images) | Your walking video frames |
| Gradient signal | ~0 (near-zero loss) | Strong (~0.85 baseline) |
| Confidence drop | 2-5% | Expected 20-40% |
| Patch visually | Barely changes from gray noise | Structured adversarial pattern |
| Mask degradation | Unmeasured | Will be measured |

**The architecture is correct. The loss function is wrong.**  
Fixing the loss function and using the right training data
are the only two changes needed to get meaningful results.

---
*Report generated as part of Gaitkeeper senior capstone project.*  
*Next notebook: `gaitkeeper_v6_topk.ipynb`*